## Importing Libraries

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.model_selection import train_test_split
import time
import sys

from thop import profile, clever_format

## Text Sequence

In [ ]:
text = """Next character prediction is a fundamental task in the field of natural language processing (NLP) that involves predicting the next character in a sequence of text based on the characters that precede it. This task is essential for various applications, including text auto-completion, spell checking, and even in the development of sophisticated AI models capable of generating human-like text.

At its core, next character prediction relies on statistical models or deep learning algorithms to analyze a given sequence of text and predict which character is most likely to follow. These predictions are based on patterns and relationships learned from large datasets of text during the training phase of the model.

One of the most popular approaches to next character prediction involves the use of Recurrent Neural Networks (RNNs), and more specifically, a variant called Long Short-Term Memory (LSTM) networks. RNNs are particularly well-suited for sequential data like text, as they can maintain information in 'memory' about previous characters to inform the prediction of the next character. LSTM networks enhance this capability by being able to remember long-term dependencies, making them even more effective for next character prediction tasks.

Training a model for next character prediction involves feeding it large amounts of text data, allowing it to learn the probability of each character's appearance following a sequence of characters. During this training process, the model adjusts its parameters to minimize the difference between its predictions and the actual outcomes, thus improving its predictive accuracy over time.

Once trained, the model can be used to predict the next character in a given piece of text by considering the sequence of characters that precede it. This can enhance user experience in text editing software, improve efficiency in coding environments with auto-completion features, and enable more natural interactions with AI-based chatbots and virtual assistants.

In summary, next character prediction plays a crucial role in enhancing the capabilities of various NLP applications, making text-based interactions more efficient, accurate, and human-like. Through the use of advanced machine learning models like RNNs and LSTMs, next character prediction continues to evolve, opening new possibilities for the future of text-based technology."""

## Creating Map

In [ ]:
chars = sorted(list(set(text)))

ix_to_char = {i: ch for i, ch in enumerate(chars)}
char_to_ix = {ch: i for i, ch in enumerate(chars)}

chars = sorted(list(set(text)))

print(f"Text length: {len(text)}")
print(f"Unique characters: {len(chars)}")
print(f"Characters: {chars}\n")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Text length: 2391
Unique characters: 45
Characters: ['\n', ' ', "'", '(', ')', ',', '-', '.', 'A', 'D', 'I', 'L', 'M', 'N', 'O', 'P', 'R', 'S', 'T', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']

Using device: cuda


# Model Definitions

## Simple RNN Model

In [ ]:
class simpleRNN(nn.Module):
    def __init__(self, inp_size, hid_size, out_size):
        super(simpleRNN, self).__init__()
        
        self.hidden_size = hid_size
        self.embedding = nn.Embedding(inp_size, hid_size,)
        self.rnn = nn.RNN(hid_size, hid_size, batch_first=True)
        self.fc = nn.Linear(hid_size, out_size)

    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.rnn(embedded)
        output = self.fc(output[:, -1, :])

        return output
    

## LSTM Model

In [ ]:
class modelLSTM(nn.Module):
    def __init__(self, inp_size, hid_size, out_size):
        super(modelLSTM, self).__init__()

        self.hidden_size = hid_size
        self.embedding = nn.Embedding(inp_size, hid_size)
        self.lstm = nn.LSTM(hid_size, hid_size, batch_first=True)
        self.fc = nn.Linear(hid_size, out_size)

    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.lstm(embedded)
        output = self.fc(output[:, -1, :])

        return output

## GRU Model

In [ ]:
class modelGRU(nn.Module):
    def __init__(self, inp_size, hid_size, out_size):
        super(modelGRU, self).__init__()

        self.hidden_size = hid_size
        self.embedding = nn.Embedding(inp_size, hid_size)
        self.gru = nn.GRU(hid_size, hid_size, batch_first=True)
        self.fc = nn.Linear(hid_size, out_size)

    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.gru(embedded)
        output = self.fc(output[:, -1, :])

        return output

# Helper Functions

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
def get_model_size(model):
    # Initializing variables
    param_size = 0
    buffer_size = 0

    # Loop for counting model size
    for param in model.parameters():
        param_size += param.nelement() * param.element_size()
    
    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()

    model_size = (param_size + buffer_size) / 1024 /1024

    return model_size

In [ ]:
def compute_flops(model, *inputs):
    flops, params, *_ = profile(model, inputs=inputs, verbose=False)
    flops, params = clever_format([flops, params], "%.3f")

    return flops, params

In [ ]:
def prep_dataset(max_len):
    X = []
    Y = []

    for i in range(len(text) - max_len):
        sequence = text[i:(i + max_len)]
        label = text[i + max_len]
        X.append([char_to_ix[char] for char in sequence])
        Y.append(char_to_ix[label])

    X = np.array(X, dtype=np.int64)
    Y = np.array(Y, dtype=np.int64)

    # Splitting into train/val
    X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=42)

    # Converting to tensors
    X_train = torch.tensor(X_train, dtype=torch.long).to(device)
    Y_train = torch.tensor(Y_train, dtype=torch.long).to(device)
    X_val = torch.tensor(X_val, dtype=torch.long).to(device)
    Y_val = torch.tensor(Y_val, dtype=torch.long).to(device)

    return X_train, X_val, Y_train, Y_val

In [ ]:
def train_model(model, X_train, X_val, Y_train, Y_val, epochs=100, lr=0.005):
    criteria = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    train_losses = []
    val_losses = []
    val_accuracies = []

    start_time = time.time()

    for epoch in range(epochs):
        # Training Loop
        model.train()
        optimizer.zero_grad()
        train_out = model(X_train)
        train_loss = criteria(train_out, Y_train)
        train_loss.backward()
        optimizer.step()
        train_losses.append(train_loss.item())

        # Validation Loop

        model.eval()
        with torch.no_grad():
            val_out = model(X_val)
            val_loss = criteria(val_out, Y_val)
            val_losses.append(val_loss.item())
            _, predicted = torch.max(val_out, 1)
            val_accuracy = (predicted == Y_val).float().mean()
            val_accuracies.append(val_accuracy.item())

    training_time = time.time() - start_time

    return {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'val_acc': val_accuracies,
        'final_train_loss': train_losses[-1],
        'final_val_loss': val_losses[-1],
        'final_val_acc': val_accuracies[-1],
        'train_time': training_time
    }

# Main Function

In [ ]:
# Hyperparameters
hidden_size = 128
epochs = 100
learning_rate = 0.005
sequence_lengths = [10, 20, 30]
model_types = ['RNN', 'LSTM', 'GRU']

results = {}

for seq in sequence_lengths:
    print(f'\n{'=' * 100}')
    print(f'Sequence Length: {seq}')
    print(f'\n{'=' * 100}')

    # Preparing the dataset
    X_train, X_val, Y_train, Y_val = prep_dataset(seq)
    
    print(f'Training Set Size: {X_train.shape[0]} | Validation Set Size: {X_val.shape[0]}')

    results[seq] = {}

    for model_type in model_types:
        print(f'\nTraining {model_type}...')

        # Creating model

        if model_type == 'RNN':
            model = simpleRNN(len(chars), hidden_size, len(chars)).to(device)
        elif model_type == 'LSTM':
            model = modelLSTM(len(chars), hidden_size, len(chars)).to(device)
        else:
            model = modelGRU(len(chars), hidden_size, len(chars)).to(device)

        # Calculating model metrics
        num_params = count_params(model)
        model_size = get_model_size(model)
    
        sample_input = torch.randint(0, len(chars), (1, seq)).to(device)
        flops, params = compute_flops(model, sample_input)

        # Commencing training
        metrics = train_model(model, X_train, X_val, Y_train, Y_val, epochs, learning_rate)

        # Printing results
        print(f'Parameters: {num_params:,}')
        print(f'Model Size: {model_size:.6f} MB')
        print(f'Model Complexity (FLOPs): {flops}')
        print(f'Final Training Loss: {metrics['final_train_loss']:.6f}')
        print(f'Final Validation Loss: {metrics['final_val_loss']:.6f}')
        print(f'Final Validation Accuracy: {metrics['final_val_acc']:.6f}')
        print(f'Training Time: {metrics['train_time']:.2f}s')


Sequence Length: 10

Training Set Size: 1904 | Validation Set Size: 477

Training RNN...
Parameters: 44,589
Model Size: 0.170094 MB
Final Training Loss: 0.080853
Final Validation Loss: 2.696371
Final Validation Accuracy: 0.505241
Training Time: 0.35s

Training LSTM...
Parameters: 143,661
Model Size: 0.548023 MB
Final Training Loss: 0.086616
Final Validation Loss: 2.508784
Final Validation Accuracy: 0.482180
Training Time: 1.32s

Training GRU...
Parameters: 110,637
Model Size: 0.422047 MB
Final Training Loss: 0.062651
Final Validation Loss: 2.580609
Final Validation Accuracy: 0.509434
Training Time: 0.93s

Sequence Length: 20

Training Set Size: 1896 | Validation Set Size: 475

Training RNN...
Parameters: 44,589
Model Size: 0.170094 MB
Final Training Loss: 0.077681
Final Validation Loss: 2.671759
Final Validation Accuracy: 0.515789
Training Time: 0.59s

Training LSTM...
Parameters: 143,661
Model Size: 0.548023 MB
Final Training Loss: 0.184791
Final Validation Loss: 2.275283
Final Valid